# Telugu LoRA vs QLoRA — Kaggle training notebook

**Stage 2:** install a pinned Axolotl into an isolated venv and run a ~20-step QLoRA smoke test
on `Qwen/Qwen3-4B-Instruct-2507`.

Kaggle settings: **Accelerator = GPU T4** · **Internet = On** · Secret `HF_TOKEN` attached.

## 1. Environment check (Kaggle's own Python / driver)

In [ ]:
!python --version
!nvidia-smi

## 2. Install Axolotl into an isolated venv

Kaggle's preinstalled torch is left untouched. `--torch-backend auto` makes uv pick the
PyTorch CUDA build that matches this machine's GPU driver.

In [ ]:
%%bash
set -e
pip install -q uv
uv venv --python 3.12 /tmp/axo-venv
PY=/tmp/axo-venv/bin/python
uv pip install --python $PY -q packaging setuptools wheel ninja
uv pip install --python $PY --torch-backend auto "torch==2.13.0"
uv pip install --python $PY --torch-backend auto --no-build-isolation "axolotl==0.20.0"

## 3. Verify the install and record versions

In [ ]:
%%bash
/tmp/axo-venv/bin/python - <<'EOF'
import torch, transformers, peft, bitsandbytes, accelerate, trl, datasets, axolotl
print("torch        ", torch.__version__, "| CUDA build", torch.version.cuda)
print("GPU          ", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
print("transformers ", transformers.__version__)
print("peft         ", peft.__version__)
print("bitsandbytes ", bitsandbytes.__version__)
print("accelerate   ", accelerate.__version__)
print("trl          ", trl.__version__)
print("datasets     ", datasets.__version__)
print("axolotl      ", axolotl.__version__)
EOF
uv pip freeze --python /tmp/axo-venv/bin/python > /kaggle/working/versions-axolotl.txt
echo "full freeze -> /kaggle/working/versions-axolotl.txt"

## 4. Secrets and GPU selection (one T4 per run, for a fair comparison)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("HF_TOKEN loaded:", bool(os.environ["HF_TOKEN"]))

## 5. Tiny smoke dataset

8 hand-written en→te pairs in the same chat `messages` format the real dataset will use
(Stage 3). Only checks that the pipeline runs — not for learning anything.

In [ ]:
import json

pairs = [
    ("Good morning.", "శుభోదయం."),
    ("How are you?", "మీరు ఎలా ఉన్నారు?"),
    ("I am going to school.", "నేను పాఠశాలకు వెళ్తున్నాను."),
    ("Water is essential for life.", "జీవితానికి నీరు చాలా అవసరం."),
    ("The weather is nice today.", "ఈ రోజు వాతావరణం బాగుంది."),
    ("Please read this book.", "దయచేసి ఈ పుస్తకం చదవండి."),
    ("My mother cooks delicious food.", "మా అమ్మ రుచికరమైన భోజనం వండుతుంది."),
    ("Thank you very much.", "చాలా ధన్యవాదాలు."),
]
with open("/kaggle/working/smoke.jsonl", "w", encoding="utf-8") as f:
    for en, te in pairs:
        msgs = [
            {"role": "user", "content": f"Translate the following English text to Telugu.\n\n{en}"},
            {"role": "assistant", "content": te},
        ]
        f.write(json.dumps({"messages": msgs}, ensure_ascii=False) + "\n")
print(open("/kaggle/working/smoke.jsonl", encoding="utf-8").readline())

## 6. Smoke-test config (QLoRA, T4-safe)

fp16 (T4 has no bf16) · SDPA attention (no flash-attn on T4) · gradient checkpointing on.

In [ ]:
%%writefile /kaggle/working/smoke-qlora.yml
base_model: Qwen/Qwen3-4B-Instruct-2507

# QLoRA: 4-bit NF4 base (Axolotl defaults: nf4 + double quant)
load_in_4bit: true
adapter: qlora
lora_r: 16
lora_alpha: 32
lora_dropout: 0.05
lora_target_linear: true

chat_template: tokenizer_default
datasets:
  - path: /kaggle/working/smoke.jsonl
    ds_type: json
    type: chat_template
    field_messages: messages
    roles_to_train: ["assistant"]
dataset_prepared_path: /kaggle/working/last_run_prepared
val_set_size: 0

sequence_len: 512
sample_packing: false

# T4 constraints
fp16: true
bf16: false
tf32: false
attn_implementation: sdpa
gradient_checkpointing: true

micro_batch_size: 1
gradient_accumulation_steps: 4
max_steps: 20
learning_rate: 0.0002
optimizer: paged_adamw_8bit
seed: 42

logging_steps: 1
save_strategy: "no"
use_tensorboard: true
output_dir: /kaggle/working/outputs/smoke-qlora

## 7. Run the smoke test (~20 steps)

In [ ]:
%%bash
# axolotl calls `accelerate launch` from PATH -> put the venv first, or Kaggle's system accelerate runs.
export PATH=/tmp/axo-venv/bin:$PATH
export CUDA_VISIBLE_DEVICES=0
# args after `--` go to accelerate: force ONE process (Kaggle has 2 T4s; we want one per run)
axolotl train /kaggle/working/smoke-qlora.yml -- --num_processes 1

## 8. Check outputs

In [ ]:
!ls -la /kaggle/working/outputs/smoke-qlora
!find /kaggle/working/outputs/smoke-qlora -name "events.out.tfevents*"